# Réseau piéton

In [ ]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads Open data for the Canton of Geneva. 
# Only run section 5. to dowload data and save the network into segments
# -------------------------------------------------

import osmnx as ox
import os
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import LineString
from shapely.ops import nearest_points
from shapely.ops import unary_union
from shapely.geometry import Point
import networkx as nx
from shapely.ops import substring

from tqdm import tqdm

import math
import geopandas as gpd
from shapely.geometry import LineString
from shapely.strtree import STRtree

## Segmentation du réseau et export du fichier

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE or GGminGE
network = "walk"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-1'

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-1'

if territory == 'GGminGE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GGminGE/step-1'
    output_step2_path='../../Data/output/GGminGE/step-2'
    output_step3_path='../../Data/output/GGminGE/step-3'


    save_path = '../../Data/output/GGminGE/step-1'

In [ ]:
if territory == 'GG':
    if network == "walk":
        nodes_graph_path = f"{input_file_path}/networkGG/walk_nodes_graph.geojson"
        edges_graph_path = f"{input_file_path}/networkGG/walk_edges_graph.geojson"

    if network == "bike":
        nodes_graph_path = f"{input_file_path}/networkGG/bike_nodes_graph.geojson"
        edges_graph_path = f"{input_file_path}/networkGG/bike_edges_graph.geojson"

if territory == 'GE':
    if network == "walk":
        edges_graph_path = f"{input_file_path}/attributs/GE/network_couche_OCT.shp/RP_final.shp"
    if network == "bike":
        print("Bike network for GE not available yet.")

if territory == 'GGminGE':
    if network == "walk":
        print("Pedestrian network for GG minus GE not available yet.")
    if network == "bike":
        edges_graph_path = f"{input_file_path}/networkGGminGE/bike_edges_graph.geojson"



def fetch_network(network: str, path: str, operation_crs: str = "EPSG:2056"):
    """
    Load walk or bike network (edges) as GeoDataFrame and project to operation_crs.

    Parameters
    ----------
    network : str
        "walk" or "bike" (only for semantic check)
    path : str
        Path to the network file (geojson/shp)
    operation_crs : str
        Target CRS (default: EPSG:2056)

    Returns
    -------
    GeoDataFrame
    """
    if network not in {"walk", "bike"}:
        raise ValueError("network must be 'walk' or 'bike'")

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    net = gpd.read_file(path)

    # If the file has no CRS, you MUST set it manually before projecting
    if net.crs is None:
        raise ValueError(
            f"{path} has no CRS. Set it first, e.g.: net = net.set_crs('EPSG:4326')"
        )

    net = net.to_crs(operation_crs)
    return net

In [ ]:
#import network
print("Loading segments, replace file path -->")
if territory =='GG':
    net = fetch_network(
        network,
        path=edges_graph_path,
        operation_crs=operation_crs
    )

if territory =='GE':
    net = fetch_network(
        network,
        path=edges_graph_path,
        operation_crs=operation_crs
    )


In [ ]:
net.head()

**Stats on the network**

In [ ]:
'''# Add Length column in meters
net['length_m'] = net.geometry.length

# DETAILED NETWORK STATISTICS
print("\n" + "="*100)
print("DETAILED PEDESTRIAN NETWORK STATISTICS - GENEVA")
print("="*100)

# Calculate total length once for percentage calculations
total_length_km = net['length_m'].sum() / 1000

# 1. BASIC NETWORK INFORMATION
print("\n1. BASIC NETWORK INFORMATION")
print("-" * 100)
print(f"Total number of edges: {len(net):,}")
print(f"Total network length: {total_length_km:.2f} km")
print(f"Mean edge length: {net['length_m'].mean():.2f} m")
print(f"Median edge length: {net['length_m'].median():.2f} m")

# 2. SIDEWALK WIDTH DISTRIBUTION (Largeur)
print("\n2. SIDEWALK WIDTH DISTRIBUTION (Largeur)")
print("-" * 100)
n_missing = net['Largeur'].isna().sum() | (net['Largeur'] == 'None').sum() | (net['Largeur'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Largeur'].isna() | (net['Largeur'] == 'None') | (net['Largeur'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':20s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for width, count in net['Largeur'].value_counts(dropna=False).items():
    if pd.isna(width) or str(width) in ['None', '', 'nan']:
        continue
    length_km = net[net['Largeur'] == width]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {width:20s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 3. INFRASTRUCTURE TYPE (Objet)
print("\n3. INFRASTRUCTURE TYPE (Objet)")
print("-" * 100)
n_missing = net['Objet'].isna().sum() | (net['Objet'] == 'None').sum() | (net['Objet'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Objet'].isna() | (net['Objet'] == 'None') | (net['Objet'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':30s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for obj_type, count in net['Objet'].value_counts(dropna=False).items():
    if pd.isna(obj_type) or str(obj_type) in ['None', '', 'nan']:
        continue
    length_km = net[net['Objet'] == obj_type]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(obj_type):30s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 4. SURFACE TYPE (Revetement)
print("\n4. SURFACE TYPE (Revetement)")
print("-" * 100)
n_missing = net['Revetement'].isna().sum() | (net['Revetement'] == 'None').sum() | (net['Revetement'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Revetement'].isna() | (net['Revetement'] == 'None') | (net['Revetement'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':30s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for surface, count in net['Revetement'].value_counts(dropna=False).items():
    if pd.isna(surface) or str(surface) in ['None', '', 'nan']:
        continue
    length_km = net[net['Revetement'] == surface]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(surface):30s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 5. MIXED USE WITH BIKES (Partage_us)
print("\n5. MIXED USE WITH BIKES (Partage_us)")
print("-" * 100)
n_missing = net['Partage_us'].isna().sum() | (net['Partage_us'] == 'None').sum() | (net['Partage_us'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Partage_us'].isna() | (net['Partage_us'] == 'None') | (net['Partage_us'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':40s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for usage, count in net['Partage_us'].value_counts(dropna=False).items():
    if pd.isna(usage) or str(usage) in ['None', '', 'nan']:
        continue
    length_km = net[net['Partage_us'] == usage]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(usage):40s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 6. SPEED LIMITS (Vitesse)
print("\n6. ADJACENT ROAD SPEED LIMITS (Vitesse)")
print("-" * 100)
n_missing = net['Vitesse'].isna().sum()
if n_missing > 0:
    length_missing = net[net['Vitesse'].isna()]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  ⚠️  NON RENSEIGNÉ: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for speed, count in net['Vitesse'].value_counts(dropna=False).sort_index().items():
    if pd.isna(speed):
        continue
    length_km = net[net['Vitesse'] == speed]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {speed:3d} km/h: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 7. PEDESTRIAN CROSSINGS WITH TRAFFIC LIGHTS (PP_Feux)
print("\n7. PEDESTRIAN CROSSINGS WITH TRAFFIC LIGHTS (PP_Feux)")
print("-" * 100)
n_missing = net['PP_Feux'].isna().sum() | (net['PP_Feux'] == 'None').sum() | (net['PP_Feux'] == '').sum()
if n_missing > 0:
    length_missing = net[net['PP_Feux'].isna() | (net['PP_Feux'] == 'None') | (net['PP_Feux'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':10s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for lights, count in net['PP_Feux'].value_counts(dropna=False).items():
    if pd.isna(lights) or str(lights) in ['None', '', 'nan']:
        continue
    length_km = net[net['PP_Feux'] == lights]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(lights):10s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 8. SLOPE STATISTICS (Pente)
print("\n8. SLOPE STATISTICS (Pente in %)")
print("-" * 100)
n_missing = net['Pente'].isna().sum()
if n_missing > 0:
    length_missing = net[net['Pente'].isna()]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"⚠️  NON RENSEIGNÉ: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)\n")

print(f"  Mean slope: {net['Pente'].mean():.2f}%")
print(f"  Median slope: {net['Pente'].median():.2f}%")
print(f"  Max slope: {net['Pente'].max():.2f}%")
print(f"  Min slope: {net['Pente'].min():.2f}%")

# Slope categories
slope_categories = [
    ('Flat (< 2%)', net['Pente'] < 2),
    ('Gentle (2-5%)', (net['Pente'] >= 2) & (net['Pente'] < 5)),
    ('Moderate (5-8%)', (net['Pente'] >= 5) & (net['Pente'] < 8)),
    ('Steep (>= 8%)', net['Pente'] >= 8)
]

print("\nSlope categories:")
for cat_name, mask in slope_categories:
    count = mask.sum()
    length_km = net[mask]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {cat_name:20s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 9. ROAD CLASS (Classe)
print("\n9. ROAD CLASS DISTRIBUTION (Classe)")
print("-" * 100)
n_missing = net['Classe'].isna().sum() | (net['Classe'] == 'None').sum() | (net['Classe'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Classe'].isna() | (net['Classe'] == 'None') | (net['Classe'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':20s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for road_class, count in net['Classe'].value_counts(dropna=False).items():
    if pd.isna(road_class) or str(road_class) in ['None', '', 'nan']:
        continue
    length_km = net[net['Classe'] == road_class]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(road_class):20s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 10. MODAL ZONES (Zone_mod)
print("\n10. MODAL ZONES (Zone_mod)")
print("-" * 100)
n_missing = net['Zone_mod'].isna().sum() | (net['Zone_mod'] == 'None').sum() | (net['Zone_mod'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Zone_mod'].isna() | (net['Zone_mod'] == 'None') | (net['Zone_mod'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':20s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for zone, count in net['Zone_mod'].value_counts(dropna=False).items():
    if pd.isna(zone) or str(zone) in ['None', '', 'nan']:
        continue
    length_km = net[net['Zone_mod'] == zone]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(zone):20s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 11. TOP 20 COMMUNES BY NETWORK LENGTH
print("\n11. TOP 20 COMMUNES BY NETWORK LENGTH")
print("-" * 100)
commune_stats = net.groupby('Commune').agg({
    'length_m': ['count', 'sum']
}).round(2)
commune_stats.columns = ['count', 'total_length_m']
commune_stats['total_length_km'] = commune_stats['total_length_m'] / 1000
commune_stats = commune_stats.sort_values('total_length_km', ascending=False).head(20)

for idx, (commune, row) in enumerate(commune_stats.iterrows(), 1):
    pct = (row['total_length_km'] / total_length_km) * 100
    print(f"  {idx:2d}. {commune:30s}: {int(row['count']):6,} edges | {row['total_length_km']:8.2f} km ({pct:5.1f}%)")

# 12. INFRASTRUCTURE TYPE (Type)
print("\n12. INFRASTRUCTURE TYPE DETAILS (Type)")
print("-" * 100)
n_missing = net['Type'].isna().sum() | (net['Type'] == 'None').sum() | (net['Type'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Type'].isna() | (net['Type'] == 'None') | (net['Type'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':30s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for infra_type, count in net['Type'].value_counts(dropna=False).items():
    if pd.isna(infra_type) or str(infra_type) in ['None', '', 'nan']:
        continue
    length_km = net[net['Type'] == infra_type]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(infra_type):30s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

# 13. CROSSINGS (Franchisse)
print("\n13. CROSSING TYPES (Franchisse)")
print("-" * 100)
n_missing = net['Franchisse'].isna().sum() | (net['Franchisse'] == 'None').sum() | (net['Franchisse'] == '').sum()
if n_missing > 0:
    length_missing = net[net['Franchisse'].isna() | (net['Franchisse'] == 'None') | (net['Franchisse'] == '')]['length_m'].sum() / 1000
    pct_missing_edges = (n_missing / len(net)) * 100
    pct_missing_length = (length_missing / total_length_km) * 100
    print(f"  {'⚠️  NON RENSEIGNÉ':30s}: {n_missing:6,} edges ({pct_missing_edges:5.1f}%) | {length_missing:8.2f} km ({pct_missing_length:5.1f}%)")

for crossing, count in net['Franchisse'].value_counts(dropna=False).items():
    if pd.isna(crossing) or str(crossing) in ['None', '', 'nan']:
        continue
    length_km = net[net['Franchisse'] == crossing]['length_m'].sum() / 1000
    pct_edges = (count / len(net)) * 100
    pct_length = (length_km / total_length_km) * 100
    print(f"  {str(crossing):30s}: {count:6,} edges ({pct_edges:5.1f}%) | {length_km:8.2f} km ({pct_length:5.1f}%)")

print("\n" + "="*100)
print("END OF DETAILED STATISTICS")
print("="*100)'''

**Segment the network**

In [ ]:
# Explode osmid lists so each row has a single osmid (works for lists, numpy arrays, and stringified lists)
import numpy as np
import ast
def to_osmid_list(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, np.ndarray)):
                return list(val)
            else:
                return [val]
        except Exception:
            return [x]
    return [x]

if "osmid" in net.columns:
    net["osmid"] = net["osmid"].apply(to_osmid_list)
    net = net.explode("osmid").reset_index(drop=True)

In [ ]:
len(net)

In [ ]:
from shapely.geometry import LineString, MultiLineString
from shapely.ops import substring
import geopandas as gpd
import pandas as pd
from tqdm import tqdm

def split_linestring(geom, segment_length):
    """Split a LineString into segments of a given length."""
    if geom.length <= segment_length:
        return [geom]
    segments = []
    start = 0.0
    while start < geom.length:
        end = min(start + segment_length, geom.length)
        seg = substring(geom, start, end)
        segments.append(seg)
        start = end
    return segments

def split_row(row, segment_length):
    geom = row.geometry
    if geom.is_empty:
        return []
    if isinstance(geom, LineString):
        segments = split_linestring(geom, segment_length)
    elif isinstance(geom, MultiLineString):
        segments = []
        for part in geom.geoms:
            segments.extend(split_linestring(part, segment_length))
    else:
        return []
    split_rows = []
    for seg in segments:
        new_row = row.copy()
        new_row.geometry = seg
        new_row["length"] = seg.length
        split_rows.append(new_row)
    return split_rows

# Desired segment length in meters
segment_length = 50

print("Splitting LineStrings into fixed-length segments...")
split_rows = []
for idx, row in tqdm(net.iterrows(), total=len(net)):
    split_rows.extend(split_row(row, segment_length))

all_segments = gpd.GeoDataFrame(split_rows, crs=net.crs)
all_segments.reset_index(drop=True, inplace=True)
# Generate segment_id as osmid + incremental number per osmid
if "osmid" in all_segments.columns:
    all_segments["segment_id"] = None
    from collections import defaultdict
    osmid_counters = defaultdict(int)
    for idx, row in all_segments.iterrows():
        osm = str(row["osmid"])
        osmid_counters[osm] += 1
        all_segments.at[idx, "segment_id"] = f"{osm}_{str(osmid_counters[osm]).zfill(3)}"
else:
    all_segments["segment_id"] = all_segments.index.astype(str).str.zfill(6)

In [ ]:
all_segments

## Suppression de segments

Dans cette section, on supprime des segments qui se confondent avec des routes départementales sur le territoire du Grand Genève. Actuellement, on supprime les segments sur les routes départementales de Haute-Savoie.  
Ce processus analyse toutes les routes données, établis un buffer (`BUFFER_DIST`) autour de chaque route, pour capter les segments de l'indice de marchabilité qui sont dans ce buffer, ensuite ils comparent leur orientation pour voir s'ils sont colinéaires ou pas (afin de ne pas prendre en compte les segments perpendiculaires aux routes qui tomberaient tout de même dans le buffer des routes). La variation d'angle autorisé est donnée par `ANGLE_THRESHOLD`, puis on vérifie aussi que la portion de route et le segments de l'indice sont suffisament colinéaire via `MIN_OVERLAP_RATIO`.

In [ ]:
# --- Paramètres ---
ANGLE_THRESHOLD   = 15
BUFFER_DIST       = 2
MIN_OVERLAP_RATIO = 0.8

# --- Chemins des routes ---
route_paths = [
    f"{input_file_path}/networkGG/SEGMENTS_A_SUPPR/ROUTES_DEPARTEMENTALES_FR/HAUTE_SAVOIE/RD_en_&_hors_agglo/rd en agglo haute savoie.shp",
    f"{input_file_path}/networkGG/SEGMENTS_A_SUPPR/ROUTES_DEPARTEMENTALES_FR/HAUTE_SAVOIE/RD_en_&_hors_agglo/rd hors agglo haute savoie.shp",
]

# --- Fonctions utilitaires ---
def segment_angles(geom):
    coords = list(geom.coords)
    segments = []
    for i in range(len(coords) - 1):
        dx = coords[i+1][0] - coords[i][0]
        dy = coords[i+1][1] - coords[i][1]
        length = math.sqrt(dx**2 + dy**2)
        angle = math.degrees(math.atan2(dy, dx)) % 180
        segments.append((angle, length))
    return segments

def angle_diff(a1, a2):
    diff = abs(a1 - a2) % 180
    return min(diff, 180 - diff)

def get_parts(geom):
    if geom.geom_type.startswith("Multi"):
        return list(geom.geoms)
    return [geom]

# --- Réinitialisation de l'index ---
all_segments = all_segments.reset_index(drop=True)

# --- Index spatial ---
indice_geoms = list(all_segments.geometry)
indice_tree  = STRtree(indice_geoms)

# --- Accumulateurs ---
colineaire_length = {i: 0.0  for i in range(len(all_segments))}
counted_segments  = {i: set() for i in range(len(all_segments))}

# --- Chargement des routes ---
route_gdfs = [gpd.read_file(p).to_crs(operation_crs) for p in route_paths]

# --- Boucle sur les routes ---
for route_gdf in route_gdfs:
    for _, route_row in route_gdf.iterrows():
        geom_route = route_row.geometry
        for part in get_parts(geom_route):
            coords = list(part.coords)
            for i in range(len(coords) - 1):
                dx = coords[i+1][0] - coords[i][0]
                dy = coords[i+1][1] - coords[i][1]
                route_seg_angle = math.degrees(math.atan2(dy, dx)) % 180
                seg_geom   = LineString([coords[i], coords[i+1]])
                seg_buffer = seg_geom.buffer(BUFFER_DIST)
                candidates = indice_tree.query(seg_buffer)
                for cid in candidates:
                    geom_indice = indice_geoms[cid]
                    if not seg_buffer.intersects(geom_indice):
                        continue
                    indice_segs = segment_angles(geom_indice)
                    for seg_idx, (seg_angle, seg_length) in enumerate(indice_segs):
                        if seg_idx in counted_segments[cid]:
                            continue
                        if angle_diff(seg_angle, route_seg_angle) < ANGLE_THRESHOLD:
                            colineaire_length[cid] += seg_length
                            counted_segments[cid].add(seg_idx)

# Sauvegarder avant filtrage pour l'export des supprimés si besoin
all_segments_before_filter = all_segments.copy()

# --- Décision finale ---
colineaires_idx = []
for i, row in all_segments.iterrows():
    total_length = row.geometry.length
    ratio = colineaire_length[i] / total_length if total_length > 0 else 0
    if ratio >= MIN_OVERLAP_RATIO:
        colineaires_idx.append(i)

# --- Récupération des segment_id supprimés ---
suppressed_ids = all_segments.loc[colineaires_idx, "segment_id"].tolist()
print(f"Segments supprimés ({len(suppressed_ids)}) : {suppressed_ids}")

# --- Mise à jour de all_segments ---
all_segments = all_segments.drop(index=colineaires_idx).reset_index(drop=True)
print(f"Segments restants : {len(all_segments)}")

# EXPORTS

In [ ]:

#save to gpkg file to save_path/"step1_all_segments.gpkg"
if not os.path.exists(save_path):
    os.makedirs(save_path)
output_file_gpkg = os.path.join(save_path, "step1_all_segments.gpkg")
all_segments.to_crs(target_crs).to_file(output_file_gpkg, driver="GPKG")
print(f"Segments saved to {output_file_gpkg}")
#save to geoparquet file
output_file_parquet = os.path.join(save_path, "step1_all_segments.parquet")
all_segments.to_crs(target_crs).to_parquet(output_file_parquet)
print(f"Segments saved to {output_file_parquet}")
